# FNN Gerçek Dünya Örneği: Müşteri Kaybı (Churn) Tahmini

## Senaryo

Bir **telekomünikasyon şirketinin** veri bilimcisi olduğunu düşün.  
Şirketin en büyük sorunu: müşteriler rakip firmaya geçiyor. Buna **churn** (müşteri kaybı) denir.

**Sorun:** Bir müşteri aboneliğini iptal etmeden önce şirkete haber vermez.  
**Çözüm:** FNN ile hangi müşterilerin kaybolma riski taşıdığını **önceden tahmin et**, onlara özel indirim veya teklif sun.

## Veri Seti: Telco Customer Churn

- **7043 müşteri** kaydı
- **20 özellik**: sözleşme türü, aylık ücret, internet servisi, müşteri süresi...
- **Hedef**: `Churn` → Yes/No (müşteri ayrıldı mı?)

## Neden FNN?

Veriler **tablolu/sayısal** (yaş, ücret, süre gibi özellikler).  
Görüntü yok, zaman serisi yok — sadece her müşteri için bir satır özellik.  
Bu tam olarak FNN'nin güçlü olduğu alan.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

print(f"TensorFlow: {tf.__version__}")

# --- nn3d: agi tarayicida canli 3D izlemek icin ---------------------------
import sys, pathlib
if not any(pathlib.Path(p, "nn3d").is_dir() for p in sys.path):
    sys.path.insert(0, str(pathlib.Path.cwd().parents[1] / "src"))
import nn3d


## Adım 1: Veriyi Yükle ve Keşfet

In [ ]:
# CSV dosyasını bul. Bu notebook nn3d deposuna kopyalandığı için veri
# dosyası yanında değil; birkaç aday yolu sırayla deniyoruz.
from pathlib import Path
ADAYLAR = [
    Path('telco_customer_churn.csv'),                              # yanında
    Path('../telco_customer_churn.csv'),                           # bir üstte
    Path('../../../yapayzekakursu/telco_customer_churn.csv'),      # kurs deposu
]
for CSV_YOLU in ADAYLAR:
    if CSV_YOLU.exists():
        break
else:
    raise FileNotFoundError(
        'telco_customer_churn.csv bulunamadı. Denenen yollar:\n  '
        + '\n  '.join(str(p.resolve()) for p in ADAYLAR)
        + '\nDosyayı bu notebook’un yanına kopyala.'
    )

df = pd.read_csv(CSV_YOLU)
print(f'Veri kaynağı: {CSV_YOLU.resolve()}')

print(f"Veri boyutu: {df.shape}  → {df.shape[0]} müşteri, {df.shape[1]} sütun")
print(f"\nSütunlar:")
print(df.columns.tolist())
print(f"\nİlk 3 satır:")
df.head(3)

In [ ]:
# Churn dağılımı — kaç müşteri ayrılmış?
churn_counts = df['Churn'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar grafiği
axes[0].bar(['Kaldı (No)', 'Ayrıldı (Yes)'], churn_counts.values, 
            color=['#4CAF50', '#F44336'])
axes[0].set_title('Müşteri Kaybı Dağılımı')
axes[0].set_ylabel('Müşteri Sayısı')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontweight='bold')

# Aylık ücret vs churn
df.boxplot(column='MonthlyCharges', by='Churn', ax=axes[1], 
           patch_artist=True)
axes[1].set_title('Aylık Ücret vs Churn')
axes[1].set_xlabel('Churn')
axes[1].set_ylabel('Aylık Ücret ($)')
plt.suptitle('')

plt.tight_layout()
plt.show()

print(f"Kalan müşteri: {churn_counts['No']} (%{churn_counts['No']/len(df)*100:.1f})")
print(f"Ayrılan müşteri: {churn_counts['Yes']} (%{churn_counts['Yes']/len(df)*100:.1f})")
print("\nDikkat: Veri dengesiz! Ayrılanlar azınlıkta. Bu önemli bir gerçek dünya problemidir.")

## Adım 2: Veri Ön İşleme

Gerçek veri **kirlidir**. Modele vermeden önce temizlememiz gerekir:

1. **Kategorik değişkenler** (Yes/No, Male/Female) → sayısal (0/1)
2. **Eksik değerler** → doldur veya at
3. **Normalizasyon** → farklı ölçeklerdeki sayıları aynı ölçeğe çek

In [ ]:
# Kopyasını alalım, orijinali bozmayalım
df_clean = df.copy()

# 1. customerID sütununu çıkar — bu sadece bir kimlik, özellik değil
df_clean.drop('customerID', axis=1, inplace=True)

# 2. TotalCharges sütununda bazı boşluklar var (string ' ') — sayıya çevir
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce')
print(f"Eksik değer sayısı: {df_clean['TotalCharges'].isna().sum()}")
# Eksik değerleri medyan ile doldur
# NOT: pandas 3'te `df['x'].fillna(..., inplace=True)` zincirli atamadir ve
# sessizce etkisiz kalir. Doğrudan sütuna atamak gerekiyor.
df_clean['TotalCharges'] = df_clean['TotalCharges'].fillna(
    df_clean['TotalCharges'].median()
)

# 3. İkili kategorik sütunları 0/1'e çevir
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 
               'PaperlessBilling', 'Churn']
mapping = {'Yes': 1, 'No': 0, 'Male': 1, 'Female': 0}
for col in binary_cols:
    df_clean[col] = df_clean[col].map(mapping)

# 4. Çok kategorili sütunları One-Hot Encode et
multi_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity', 
              'OnlineBackup', 'DeviceProtection', 'TechSupport',
              'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']
df_clean = pd.get_dummies(df_clean, columns=multi_cols, drop_first=True)

print(f"\nİşlem sonrası sütun sayısı: {df_clean.shape[1]}")
print(f"Veri boyutu: {df_clean.shape}")
df_clean.head(2)

In [ ]:
# Özellikler (X) ve hedef (y) ayır
OZELLIK_ADLARI = df_clean.drop('Churn', axis=1).columns.tolist()
X = df_clean.drop('Churn', axis=1).values.astype(np.float32)
y = df_clean['Churn'].values.astype(np.float32)

# Eğitim/test bölme
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y  
    # stratify=y: eğitim ve testte churn oranını dengeli tut
)

# Normalizasyon — StandardScaler: ortalama 0, standart sapma 1
# Neden gerekli? 'tenure' 0-72 arasında, 'TotalCharges' 0-8000 arasında.
# Bu eşitsizlik modeli yanlı hale getirir. Normalizasyon eşitler.
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)   # fit: istatistikleri hesapla, transform: uygula
X_test = scaler.transform(X_test)         # Sadece transform! Test verisine fit YAPMA.

print(f"Eğitim seti: {X_train.shape}")
print(f"Test seti:   {X_test.shape}")
print(f"Özellik sayısı: {X_train.shape[1]}")

## Adım 3: FNN Modelini Oluştur

Bu sefer **ikili sınıflandırma** yapıyoruz: müşteri ayrılır mı, ayrılmaz mı?  
Çıkış katmanında 1 nöron + **sigmoid** aktivasyonu kullanıyoruz.

Sigmoid: 0-1 arasında bir olasılık verir.  
- 0.7 → %70 ihtimalle ayrılacak  
- 0.2 → %20 ihtimalle ayrılacak (yani büyük ihtimalle kalmaya devam edecek)

**Dropout** nedir?  
Eğitim sırasında nöronların %30'unu rastgele kapatır.  
Bu, modelin tek bir yola bağımlı olmasını engeller → daha iyi genelleme.

In [ ]:
# FNN modeli — Dropout ile overfitting önleme
model = models.Sequential([
    # Giriş + İlk gizli katman
    layers.Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dropout(0.3),          # %30 nöronu rastgele kapat (sadece eğitimde)
    
    # İkinci gizli katman
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    
    # Üçüncü gizli katman
    layers.Dense(32, activation='relu'),
    
    # Çıkış: 1 nöron, sigmoid → 0-1 arası olasılık
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',   # İkili sınıflandırma için
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]  # AUC de izle
)

model.summary()

## AUC (Area Under Curve) Nedir?

Sadece accuracy (doğruluk) bakmak dengesiz veri setlerinde yanıltıcı olabilir.  
Eğer veri setinin %85'i "Kaldı" ise, model her şeye "Kaldı" dese bile %85 accuracy alır — ama işe yaramaz!

**AUC-ROC**: Modelin gerçek pozitifleri yanlış pozitiflerden ne kadar iyi ayırt edebildiğini ölçer.
- 0.5 → Rastgele tahmin (işe yaramaz)
- 1.0 → Mükemmel
- 0.8+ → İyi bir model

## Canlı 3D görselleştirme`nn3d.Monitor` bir Keras callback'i. `fit()` başlar başlamaz tarayıcıda bir sekmeaçılır ve eğitim boyunca canlı akar:- **Sol taraftaki özellik adları** (`tenure`, `MonthlyCharges`…) o anki aktivasyona  göre renklenir — sönük gri = sessiz özellik, turkuaz = yüksek aktivasyon- **Turkuaz çizgi** pozitif ağırlık, **kızıl çizgi** negatif ağırlık- Ağırlıklar öğrendikçe çizgilerin rengi ve parlaklığı değişir- Sağ üstte `loss`, `accuracy`, `auc` canlıBir nöronun üzerine gel: bütün bağlantıları altın sarısına döner, gerisi söner.

In [ ]:
# Eğitim — nn3d.Monitor ile canlı 3D izleme
# every=10: her 10 batch'te bir kare gönder. Her batch'te göndermek
# ekstra ileri yayılım demek ve eğitimi yavaşlatır.
izleyici = nn3d.Monitor(
    X_test[:1],
    every=10,
    input_labels=OZELLIK_ADLARI,
    output_labels=['Ayrılma Olasılığı'],
)

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=[izleyici],
    verbose=1
)

print("\nEğitim tamamlandı!")

In [ ]:
# Eğitim grafiklerini çiz
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, metric, title in zip(
    axes,
    ['loss', 'accuracy', 'auc'],
    ['Kayıp (Binary Crossentropy)', 'Doğruluk', 'AUC-ROC']
):
    ax.plot(history.history[metric], label='Eğitim', color='#2196F3')
    ax.plot(history.history[f'val_{metric}'], label='Test', 
            color='#FF5722', linestyle='--')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('FNN — Müşteri Kaybı Tahmini Eğitim Geçmişi', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Adım 4: Sonuçları Değerlendir

### Confusion Matrix (Karmaşıklık Matrisi)

```
                    Tahmin: Kaldı    Tahmin: Ayrıldı
Gerçek: Kaldı     True Negative    False Positive
Gerçek: Ayrıldı   False Negative   True Positive
```

- **False Negative**: Ayrılacak müşteriyi "Kalacak" dedi → en kötü hata! Fırsatı kaçırdık.
- **False Positive**: Kalacak müşteriyi "Ayrılacak" dedi → gereksiz indirim verdik ama zarar az.

In [ ]:
# Test seti üzerinde tahminler
y_pred_prob = model.predict(X_test, verbose=0).flatten()  # Olasılıklar (0-1)
y_pred = (y_pred_prob >= 0.5).astype(int)                # 0.5 eşiği ile sınıfa çevir

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix görsel
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Kaldı', 'Ayrıldı'],
            yticklabels=['Kaldı', 'Ayrıldı'], ax=axes[0])
axes[0].set_title('Confusion Matrix')
axes[0].set_ylabel('Gerçek')
axes[0].set_xlabel('Tahmin')

# Tahmin olasılık dağılımı
axes[1].hist(y_pred_prob[y_test == 0], bins=30, alpha=0.7, 
             color='#4CAF50', label='Gerçekte Kaldı')
axes[1].hist(y_pred_prob[y_test == 1], bins=30, alpha=0.7, 
             color='#F44336', label='Gerçekte Ayrıldı')
axes[1].axvline(x=0.5, color='black', linestyle='--', label='Eşik (0.5)')
axes[1].set_title('Tahmin Olasılık Dağılımı')
axes[1].set_xlabel('Ayrılma Olasılığı')
axes[1].set_ylabel('Müşteri Sayısı')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\n=== Detaylı Sınıflandırma Raporu ===")
print(classification_report(y_test, y_pred, target_names=['Kaldı', 'Ayrıldı']))

In [ ]:
# İş Etkisi: En riskli müşterileri belirle
# Gerçekte bu tablo bir pazarlama ekibine gönderilir

feature_names = df_clean.drop('Churn', axis=1).columns.tolist()
test_df = pd.DataFrame(X_test, columns=feature_names)
test_df['Gercek_Churn'] = y_test.astype(int)
test_df['Churn_Olasiligi'] = y_pred_prob
test_df['Risk_Seviyesi'] = pd.cut(
    y_pred_prob,
    bins=[0, 0.3, 0.6, 1.0],
    labels=['Düşük Risk', 'Orta Risk', 'Yüksek Risk']
)

risk_summary = test_df['Risk_Seviyesi'].value_counts()
print("=== Risk Seviyesi Dağılımı ===")
for level, count in risk_summary.items():
    print(f"{level}: {count} müşteri")

print("\n=== En Yüksek Riskli 5 Müşteri ===")
top_risk = test_df.nlargest(5, 'Churn_Olasiligi')[['Churn_Olasiligi', 'Risk_Seviyesi', 'Gercek_Churn']]
top_risk['Churn_Olasiligi'] = top_risk['Churn_Olasiligi'].apply(lambda x: f"%{x*100:.1f}")
print(top_risk.to_string())

## Sonuç

Bu notebook'ta gerçek bir iş problemi çözdük:

1. **Ham CSV** verisini modele hazır hale getirdik (temizleme, encoding, normalizasyon)
2. **FNN** ile müşteri kaybını tahmin ettik
3. **Confusion matrix** ve **AUC** ile modeli değerlendirdik
4. Yüksek riskli müşterileri belirleyerek **iş değeri** ürettik

### Gerçek Hayatta Ne Olur?

Bu modelin çıktısı bir CRM (Müşteri İlişkileri Yönetimi) sistemine entegre edilir.  
"Yüksek Risk" listesindeki müşterilere otomatik olarak özel teklifler gönderilir.  
Şirket, kaybettiği her müşteri için ortalama $1000 tasarruf eder.

### Precision vs Recall Dengesi

- **Precision yüksek, Recall düşük**: Çok az müşteriye teklif yaptık ama hepsini doğru hedefledik
- **Recall yüksek, Precision düşük**: Çok müşteriye teklif yaptık, bazıları gereksizdi ama hiçbirini kaçırmadık

İş stratejisine göre 0.5 eşiği değiştirilebilir!